# Case Palavritas - the news\n## Analista de Dados Produto & Growth

In [ ]:
import sys\nsys.path.insert(0, '..')\n\nimport pandas as pd\nimport numpy as np\nfrom src.load import load_sessions, load_attempts, load_user_profile, save_cleaned\nfrom src.cleaning import rodar_diagnostico\nfrom src.analysis import (\n    retencao_por_resultado, retencao_por_streak, retencao_por_hora,\n    retencao_por_device, retencao_por_newsletter, dificuldade_por_palavra,\n    juntar_sessions_profile, retencao_por_perfil, padrao_tentativas,\n    matriz_correlacao, tabela_features, analise_coorte, curva_retencao,\n    retencao_por_usuario, segmentar_usuarios\n)\nfrom src.statistics import (\n    teste_z_proporcoes, ranking_correlacao, teste_anova, teste_t,\n    ic_proporcao, correcao_bonferroni\n)\nfrom src.plots import (\n    grafico_retencao_resultado, grafico_retencao_streak, grafico_retencao_hora,\n    grafico_palavras, grafico_heatmap, grafico_perfil, grafico_ranking,\n    grafico_curva_retencao, grafico_coorte\n)

## 1. Carregamento e Diagnostico

In [ ]:
sessions, attempts, profile = rodar_diagnostico()

In [ ]:
sessions.head()

In [ ]:
attempts.head()

In [ ]:
profile.head()

## 2. Retencao por Resultado (win/lose) com IC 95%

In [ ]:
r_resultado = retencao_por_resultado(sessions)\nr_resultado

In [ ]:
grafico_retencao_resultado(r_resultado)

In [ ]:
wins_d1 = sessions[sessions['result'] == 'win']['played_next_day'].sum()\nwins_total = (sessions['result'] == 'win').sum()\nloses_d1 = sessions[sessions['result'] == 'lose']['played_next_day'].sum()\nloses_total = (sessions['result'] == 'lose').sum()\n\nz = teste_z_proporcoes(wins_d1, wins_total, loses_d1, loses_total)\nic_win = ic_proporcao(wins_d1, wins_total)\nic_lose = ic_proporcao(loses_d1, loses_total)\n\nprint('Teste Z - resultado vs D+1:')\nprint(f\"  Win D+1: {z['prop_a']:.2%}  IC95=[{ic_win['ic_inferior']:.2%}, {ic_win['ic_superior']:.2%}]\")\nprint(f\"  Lose D+1: {z['prop_b']:.2%} IC95=[{ic_lose['ic_inferior']:.2%}, {ic_lose['ic_superior']:.2%}]\")\nprint(f\"  z={z['z']:.3f}, p={z['p_valor']:.4f}, significativo={z['significativo']}\")

## 3. Retencao por Streak

In [ ]:
r_streak = retencao_por_streak(sessions)\nr_streak

In [ ]:
grafico_retencao_streak(r_streak)

## 4. Retencao por Hora do Dia

In [ ]:
r_hora = retencao_por_hora(sessions)\nr_hora

In [ ]:
grafico_retencao_hora(r_hora)

## 5. Retencao por Device

In [ ]:
r_device = retencao_por_device(sessions)\nr_device

In [ ]:
android_d30 = sessions[(sessions['device'] == 'Android')]['active_d30'].sum()\nandroid_n = (sessions['device'] == 'Android').sum()\nios_d30 = sessions[(sessions['device'] == 'iOS')]['active_d30'].sum()\nios_n = (sessions['device'] == 'iOS').sum()\n\nz = teste_z_proporcoes(android_d30, android_n, ios_d30, ios_n)\nprint('Teste Z - device vs D30:')\nprint(f\"  Android D30: {z['prop_a']:.2%}, iOS D30: {z['prop_b']:.2%}\")\nprint(f\"  z={z['z']:.3f}, p={z['p_valor']:.4f}, significativo={z['significativo']}\")

## 6. Newsletter e Retencao

In [ ]:
r_news = retencao_por_newsletter(sessions)\nr_news

In [ ]:
open_d1 = sessions[sessions['newsletter_open_before_game'] == True]['played_next_day'].sum()\nopen_n = (sessions['newsletter_open_before_game'] == True).sum()\nno_open_d1 = sessions[sessions['newsletter_open_before_game'] == False]['played_next_day'].sum()\nno_open_n = (sessions['newsletter_open_before_game'] == False).sum()\n\nz = teste_z_proporcoes(open_d1, open_n, no_open_d1, no_open_n)\nic_open = ic_proporcao(open_d1, open_n)\nic_no = ic_proporcao(no_open_d1, no_open_n)\n\nprint('Teste Z - newsletter vs D+1:')\nprint(f\"  Abriu: {z['prop_a']:.2%} IC95=[{ic_open['ic_inferior']:.2%}, {ic_open['ic_superior']:.2%}]\")\nprint(f\"  Nao abriu: {z['prop_b']:.2%} IC95=[{ic_no['ic_inferior']:.2%}, {ic_no['ic_superior']:.2%}]\")\nprint(f\"  z={z['z']:.3f}, p={z['p_valor']:.4f}, significativo={z['significativo']}\")

## 7. Dificuldade por Palavra

In [ ]:
r_palavras = dificuldade_por_palavra(sessions)\nr_palavras.head(15)

In [ ]:
grafico_palavras(r_palavras)

## 8. Perfil dos Jogadores (merge com user_profile)

In [ ]:
merged = juntar_sessions_profile(sessions, profile)\nprint(f'Sessoes com perfil: {len(merged)} de {len(sessions)}')\nprint(f'Jogadores com perfil: {merged[\"user_id\"].nunique()} de {sessions[\"user_id\"].nunique()}')

In [ ]:
for var in ['salary_range', 'sector', 'age_range', 'typical_play_time']:\n    print(f'\\n--- {var} ---')\n    r = retencao_por_perfil(merged, var)\n    print(r.head(8).to_string())\n    grafico_perfil(r, var, f'Retencao por {var}')

In [ ]:
print('ANOVA - salary_range vs D30:')\nr = teste_anova(merged, 'salary_range', 'active_d30')\nprint(f\"  F={r['f']:.3f}, p={r['p_valor']:.4f}, significativo={r['significativo']}\")\n\nprint('ANOVA - sector vs D30:')\nr = teste_anova(merged, 'sector', 'active_d30')\nprint(f\"  F={r['f']:.3f}, p={r['p_valor']:.4f}, significativo={r['significativo']}\")

## 9. Curva de Retencao Global

In [ ]:
curva = curva_retencao(sessions)\ncurva[curva['dia'].isin([1, 3, 7, 14, 30])]

In [ ]:
grafico_curva_retencao(curva)

## 10. Analise de Coorte Semanal

In [ ]:
cohort = analise_coorte(sessions)\ncohort.head(20)

In [ ]:
grafico_coorte(cohort)

## 11. Analise a Nivel de Usuario e Segmentacao

In [ ]:
user_metrics = retencao_por_usuario(sessions)\nuser_metrics = segmentar_usuarios(user_metrics)\nuser_metrics['segmento'].value_counts()

In [ ]:
user_metrics.groupby('segmento').agg(\n    usuarios=('user_id', 'count'),\n    d1_medio=('d1_medio', 'mean'),\n    d30_medio=('d30_medio', 'mean'),\n    win_rate_medio=('win_rate', 'mean'),\n    streak_max_medio=('streak_max', 'mean'),\n).round(3)

## 12. Correlacoes e Ranking de Features

In [ ]:
corr = matriz_correlacao(sessions)\ncorr

In [ ]:
grafico_heatmap(corr)

In [ ]:
ranking_d1 = ranking_correlacao(corr, 'played_next_day')\nranking_d1

In [ ]:
ranking_d30 = ranking_correlacao(corr, 'active_d30')\nranking_d30

## 13. Features com perfil (ranking completo com correcao Bonferroni)

In [ ]:
features = tabela_features(sessions, profile)\ncorr_full = features.corr()\nranking_features_d1 = ranking_correlacao(corr_full, 'jogou_dia_seguinte')\nranking_features_d1.head(12)

In [ ]:
ranking_features_d30 = ranking_correlacao(corr_full, 'ativo_d30')\nranking_features_d30.head(12)

In [ ]:
grafico_ranking(ranking_features_d1, 'O que mais se associa com jogar no dia seguinte?')

## 14. Protocolo de Tentativas

In [ ]:
padrao = padrao_tentativas(attempts, sessions)\npadrao

## 15. Salvar dados limpos

In [ ]:
save_cleaned(sessions, attempts, profile)